> **¿Estás en Binder?** Ya está todo instalado: salta la sección de instalación
> y usa **Run → Run All Cells**. Recuerda descargar tus resultados antes de cerrar
> (la sesión se borra si queda ~10 min sin uso).

# Laboratorio ERT 2D — Estudio de sensibilidad

**Métodos Electromagnéticos** · Tomografía de Resistividad Eléctrica
Basado en el ejemplo *2D ERT modelling and inversion* de PyGIMLi.

---

## Objetivo

Entender **qué tan fielmente la inversión ERT reconstruye el subsuelo**, y de qué
depende esa fidelidad. Para eso corremos un experimento controlado: construimos
modelos sintéticos donde conocemos la respuesta verdadera, simulamos la medición,
invertimos como si no supiéramos nada, y comparamos.

| Experimento | Variable que cambia | Casos |
|---|---|---|
| **A** | Espesor de la capa que aloja el objeto | 4 m, 6 m, 10 m |
| **B** | Contraste de resistividad objeto/encajante | ×2, ×20, ÷10 |
| **C** | Geometría del objeto (área ≈ constante) | círculo, dique vertical, tabular horizontal |
| **D** | Regularizador λ | 1, 5, 20, 100, 500 |

**Principio de diseño:** en cada experimento cambia **una sola variable**. Todo lo
demás —arreglo, ruido, semilla aleatoria, malla de inversión— queda fijo. Si la
semilla del ruido cambiara entre casos, no sabríamos si la diferencia observada
es física o azar.

---

## ⚙️ Instalación

### Opción 1 — Google Colab

Corre la celda de abajo **primero**. Tarda unos 3–5 minutos y **al terminar
reinicia el entorno a propósito**: verás un mensaje de que la sesión se cayó.
Eso es esperado y correcto.

**Por qué el reinicio es obligatorio.** Instalar PyGIMLi cambia la versión de
NumPy. Pero SciPy y los demás paquetes ya cargados en memoria fueron compilados
contra la versión *anterior*, así que quedan incompatibles. El síntoma típico es
que SciPy deja de importarse y PyGIMLi avisa `scipy.sparse is not available` — y
después revienta con `'NoneType' object has no attribute 'dok_matrix'`.
Reiniciar obliga a que todo se cargue de nuevo contra la misma versión de NumPy.
Por eso la celda instala **scipy en el mismo comando** que PyGIMLi: pip resuelve
versiones mutuamente compatibles de una sola pasada.

**Después del reinicio:** `Entorno de ejecución → Ejecutar todo`. La celda
detecta que PyGIMLi ya está instalado y no vuelve a reinstalar ni a reiniciar.

### Opción 2 — Tu computador con Windows (recomendada, verificada)

En la carpeta del laboratorio hay dos archivos para doble clic:

1. **`instalar_entorno.bat`**: se corre **una sola vez**. Descarga micromamba (un
   conda liviano) y crea el entorno `ert` en `%USERPROFILE%\micromamba`. No
   necesita Anaconda ni permisos de administrador. Tarda 5–10 minutos.
2. **`abrir_laboratorio.bat`**: se usa **cada vez**. Abre JupyterLab en el
   navegador, ya dentro del entorno y en esta carpeta.

Versiones del entorno, con las que se verificaron los 14 casos dentro de Jupyter:
**Python 3.11, PyGIMLi 1.5.5, NumPy 1.26, matplotlib 3.8, OpenBLAS**.

> **Dos trampas de Windows que el instalador ya resuelve** (por si algún día
> armas el entorno a mano):
> - **El kernel muere sin mensaje** (o `code 0xc06d007f`): es la librería MKL de
>   NumPy. Se evita instalando `"libblas=*=*openblas"`.
> - **El kernel no arranca** y aparece `Bad file descriptor (...epoll.cpp:73)`: es
>   un bug del `pyzmq` de conda-forge en Windows. Se arregla con
>   `pip install --force-reinstall --no-deps pyzmq`.

En macOS o Linux, el equivalente con conda es:

```bash
conda create -n ert -c gimli -c conda-forge python=3.11 "pygimli=1.5.5" "numpy=1.26" pandas matplotlib jupyterlab
conda activate ert
jupyter lab
```

Fuera de Colab la celda de instalación no hace nada. La corrida completa
(14 inversiones) tarda unos pocos minutos.

In [ ]:
# ==================================================================
# INSTALACION - solo hace algo si estas en Google Colab
# Al terminar REINICIA el entorno a proposito. Es lo esperado.
# ==================================================================
import importlib.util
import subprocess
import sys

# OJO: find_spec("google.colab") LANZA un error (en vez de devolver None)
# cuando ni siquiera existe el paquete "google", que es lo normal fuera
# de Colab. Por eso se pregunta primero por "google".
EN_COLAB = (importlib.util.find_spec("google") is not None and
            importlib.util.find_spec("google.colab") is not None)
YA_ESTA  = importlib.util.find_spec("pygimli") is not None

if EN_COLAB and not YA_ESTA:
    print("Instalando PyGIMLi en Colab. Tarda 3-5 minutos...\n")

    # Estos paquetes preinstalados de Colab fijan versiones de NumPy que
    # chocan con las que pide PyGIMLi.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                    "numba", "tensorflow", "pytensor", "thinc"])

    # CLAVE: scipy va en el MISMO comando que pygimli, para que pip resuelva
    # numpy/scipy/pygimli de una sola pasada y queden compatibles entre si.
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "pygimli", "tetgen", "scipy"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("La instalacion reporto errores:\n", r.stderr[-3000:])

    print("\n" + "=" * 60)
    print("Instalacion lista. REINICIANDO el entorno.")
    print("Es normal y necesario que diga que la sesion se cayo.")
    print("Despues:  Entorno de ejecucion > Ejecutar todo")
    print("=" * 60)

    import os
    import time
    time.sleep(2)
    os.kill(os.getpid(), 9)      # fuerza el reinicio del runtime

elif EN_COLAB:
    print("PyGIMLi ya esta instalado. No se reinstala ni se reinicia.")
else:
    print("No estas en Colab: se usa el entorno local, no se instala nada.")

### Verificación

Esta celda comprueba que NumPy, SciPy y PyGIMLi están alineados **antes** de
correr nada pesado. Si algo falla aquí, no tiene sentido seguir.

In [ ]:
import numpy
print("numpy  ", numpy.__version__)

try:
    import scipy
    import scipy.sparse
    scipy.sparse.dok_matrix((2, 2))      # la operacion exacta que fallaba
    print("scipy  ", scipy.__version__, "- scipy.sparse OK")
except Exception as e:
    raise ImportError(
        "scipy.sparse NO funciona (%s).\n"
        "PyGIMLi lo necesita para buscar celdas y para las figuras de malla.\n"
        "Solucion: Entorno de ejecucion > Reiniciar sesion, y Ejecutar todo.\n"
        "Si persiste, corre en una celda:\n"
        "    !pip install -q --force-reinstall scipy\n"
        "y vuelve a reiniciar." % e)

## 1. El flujo de trabajo, en cuatro actos

Antes de correr nada, conviene tener claro qué hace cada etapa.

**Acto 1 — Construir el subsuelo verdadero.** Se dibuja una caja de 100 × 50 m
cortada por interfaces horizontales, y se le insertan cuerpos anómalos. Cada
región recibe una etiqueta numérica (*marker*) y a cada marker se le asigna una
resistividad. La caja es mucho más grande que la zona de medición a propósito:
así los bordes artificiales no contaminan el cálculo.

**Acto 2 — Simular la medición.** Se define un arreglo de electrodos (aquí
dipolo‑dipolo) y se resuelve por elementos finitos qué voltaje mediría cada
cuadrupla. El resultado se expresa como **resistividad aparente** $\rho_a$. Se le
suma ruido realista (1 % relativo + 1 µV de piso absoluto).

> **Detalle crítico:** hay que clavar nodos de la malla exactamente en cada
> electrodo. La corriente sale de un punto, así que el potencial tiene una
> singularidad ahí; sin refinamiento local los voltajes salen mal.

**Acto 3 — Invertir.** El programa *olvida* el modelo verdadero y busca una
distribución de resistividad que reproduzca los datos. El problema es **mal
planteado**: infinitos modelos ajustan los mismos datos. Por eso se minimiza

$$\Phi = \underbrace{\Phi_d}_{\text{ajuste a los datos}} \;+\; \lambda \underbrace{\Phi_m}_{\text{suavidad del modelo}}$$

El juez del ajuste es el **$\chi^2$**, que mide el desajuste normalizado por el
error de los datos. **$\chi^2 \approx 1$ es el ideal**: ajustaste los datos justo
hasta su nivel de ruido, ni más ni menos.

**Acto 4 — Comparar.** Modelo verdadero contra modelo invertido, en la misma
escala de color logarítmica (obligatoria en ERT: las resistividades varían por
órdenes de magnitud).

> ⚠️ Si trabajas sobre el notebook original de PyGIMLi, **borra la línea**
> `np.testing.assert_approx_equal(mgr.inv.chi2(), 0.7, significant=1)`. Es un test
> interno de los desarrolladores que verifica que *ese* ejemplo da χ²≈0.7; apenas
> cambies el modelo lanza un error y detiene la ejecución.

## 2. Configuración

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pygimli as pg
    import pygimli.meshtools as mt
    from pygimli.physics import ert
except ImportError as e:
    raise ImportError(
        "No se pudo importar PyGIMLi (%s).\n"
        "Si estas en Colab y acabas de instalarlo:\n"
        "  Entorno de ejecucion > Reiniciar sesion, y corre todo de nuevo.\n"
        "Si estas en tu maquina: activa el entorno donde instalaste PyGIMLi."
        % e)

print("PyGIMLi version:", pg.__version__)
print("NumPy version:  ", np.__version__)

In [ ]:
# ------------------------------------------------------------------
# PARAMETROS GLOBALES - identicos en todos los casos
# ------------------------------------------------------------------
OUTDIR      = "resultados_ert"

N_ELEC      = 21                 # numero de electrodos
X_ELEC      = (-15.0, 15.0)      # extremos de la linea -> espaciamiento 1.5 m
ESQUEMA     = 'dd'               # dipolo-dipolo

NOISE_LEVEL = 1                  # 1 % de ruido relativo
NOISE_ABS   = 1e-6               # 1 microvoltio de piso
SEED        = 1337               # FIJA: el mismo ruido en todos los casos

LAMBDA      = 20                 # regularizador por defecto
QUALITY     = 34                 # calidad de la malla directa

# Malla de inversion:
#   False -> la que arma sola el ERTManager a partir de los electrodos.
#            Es la opcion SEGURA y es la que usa el ejemplo oficial.
#   True  -> una grilla regular propia (mas bonita, pero hay que
#            dimensionarla con cuidado: ver la celda de la malla).
MALLA_PROPIA = False

# Mostrar el log de iteraciones de la inversion.
# Dejalo en True: es la unica forma de ver si la inversion realmente
# itero o si se quedo en el modelo inicial.
VERBOSE_INV = True

# Marcadores de region
M_CAPA1, M_CAPA2, M_BASAM, M_OBJ = 1, 2, 3, 4

os.makedirs(OUTDIR, exist_ok=True)

espaciamiento = (X_ELEC[1] - X_ELEC[0]) / (N_ELEC - 1)
print("Espaciamiento entre electrodos: %.2f m" % espaciamiento)
print("Apertura total de la linea:     %.1f m" % (X_ELEC[1] - X_ELEC[0]))
print("Profundidad de investigacion esperada: ~%.1f m" % (0.19 * (X_ELEC[1] - X_ELEC[0])))

## 3. Geometrías del objeto

Las tres formas tienen **área casi idéntica (≈ 7 m²)** a propósito. Si
comparáramos un cuerpo grande contra uno chico, la diferencia sería de volumen y
no de forma — y lo que queremos aislar aquí es el efecto de la geometría.

Un detalle que confunde a mucha gente: `mt.createCircle` con `radius=[4, 1]` **no
produce un círculo** sino una elipse (semiejes 4 y 1). En este notebook usamos un
radio escalar para que sí sea un círculo.

In [ ]:
# Las formas se definen como DATOS, no como codigo. Asi la misma definicion
# sirve para dos cosas que deben coincidir exactamente:
#   1) construir el poligono que entra a la malla
#   2) decidir analiticamente que region ocupa un punto cualquiera
OBJETOS = {
    # circulo compacto: area = pi*1.5^2 = 7.07 m2
    "circulo":     dict(tipo="circulo", centro=(0.0, -3.0), radio=1.5),
    # dique / falla: area = 1.7 x 4.0 = 6.80 m2
    "dique_vert":  dict(tipo="rect", x=(-0.85, 0.85), z=(-5.3, -1.3)),
    # lente tabular tumbado: area = 7.0 x 1.0 = 7.00 m2
    "tabular_hor": dict(tipo="rect", x=(-3.5, 3.5), z=(-3.6, -2.6)),
}

def crear_objeto(nombre):
    """Construye el poligono del objeto para meterlo en el PLC."""
    o = OBJETOS[nombre]
    if o["tipo"] == "circulo":
        # nSegments=48: con el valor por defecto (12) el "circulo" es un
        # dodecagono de 6.75 m2, no los 7.07 m2 que dice la tabla
        return mt.createCircle(pos=list(o["centro"]), radius=o["radio"],
                               nSegments=48, marker=M_OBJ,
                               boundaryMarker=10, area=0.05)
    x0, x1 = sorted(o["x"])
    z0, z1 = sorted(o["z"])          # z0 = mas profundo, z1 = mas somero
    return mt.createRectangle(start=[x0, z1], end=[x1, z0],
                              marker=M_OBJ, boundaryMarker=10, area=0.05)

def dentro_del_objeto(x, z, nombre):
    o = OBJETOS[nombre]
    if o["tipo"] == "circulo":
        cx, cz = o["centro"]
        return (x - cx) ** 2 + (z - cz) ** 2 <= o["radio"] ** 2
    x0, x1 = sorted(o["x"])
    z0, z1 = sorted(o["z"])
    return (x0 <= x <= x1) and (z0 <= z <= z1)

def marker_en(x, z, z1, z2, geometria):
    """Que region del modelo verdadero ocupa el punto (x, z).

    Se calcula de forma ANALITICA, a partir de la misma definicion
    geometrica que construyo la malla. Antes esto se hacia con
    mesh.findCell(), que internamente arma una matriz dispersa con
    scipy.sparse: innecesariamente lento y, si scipy falla, revienta.
    """
    if dentro_del_objeto(x, z, geometria):
        return M_OBJ
    if z > z1:
        return M_CAPA1
    if z > z2:
        return M_CAPA2
    return M_BASAM

GEOMETRIAS = list(OBJETOS.keys())

# Vista previa de las tres formas (todas caben dentro de la capa 2)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, nombre in zip(axes, GEOMETRIAS):
    pg.show(mt.createWorld(start=[-10, 0], end=[10, -9],
                           layers=[-1, -7], worldMarker=True)
            + crear_objeto(nombre),
            ax=ax, hold=True)
    ax.set_title(nombre)
    ax.set_xlim(-8, 8); ax.set_ylim(-9, 0)
fig.tight_layout()
plt.show()

## 4. La función núcleo

`correr_caso()` hace el ciclo completo de un caso: construye el modelo verdadero,
simula, invierte, calcula métricas y guarda la figura comparativa. En la figura
de la inversión se dibujan con **línea discontinua** el contorno del objeto
verdadero y, con **línea punteada**, las interfaces reales. Así se ve de un
vistazo cuánto se corrió, se acható o se esparció cada cosa.

### ⚠️ Cómo se ve una inversión que falló

Una inversión puede **no iterar** y devolver el modelo inicial (un semiespacio
homogéneo con la mediana de los datos) sin lanzar ningún error. Las señales son
inconfundibles:

| Señal | Qué significa |
|:--|:--|
| `rugosidad = 0.00` | El modelo no tiene ninguna estructura: es todo el mismo valor |
| `contraste_recuperado = 1.00` | Objeto y encajante salieron idénticos |
| `z_centroide_recuperado = nan` | No hay ninguna anomalía que localizar |
| `chi2` de cientos | Es el desajuste de un semiespacio homogéneo |
| El mismo `chi2` para λ distintos | λ no cambió nada porque no hubo iteraciones |

La causa más común es una **malla de inversión mal dimensionada**: pedir muchas
más incógnitas que datos, sobre todo a profundidades donde el arreglo no tiene
sensibilidad. El problema queda tan mal condicionado que la búsqueda de línea no
encuentra mejora en el primer paso, y PyGIMLi aborta devolviendo el modelo de
partida.

Por eso dos cosas en este notebook: `VERBOSE_INV = True` para ver el log de
iteraciones, y la función `revisar_inversion()`, que marca el caso como no
válido y lo grita en pantalla en vez de dejar pasar números con pinta de buenos.

### Las métricas y qué significan

| Métrica | Qué mide | Cómo leerla |
|---|---|---|
| `chi2` | Desajuste normalizado por el error de los datos | ≈ 1 es lo sano. ≫1 no ajustó; <1 sobreajustó (le creyó al ruido) |
| `rrms_pct` | Error cuadrático medio relativo | Debe quedar cerca del nivel de ruido (1 %) |
| `contraste_recuperado` | $\rho_{obj}/\rho_{enc}$ en el modelo **invertido** | Siempre más cerca de 1 que el real: la regularización suaviza |
| `recuperacion_log` | $\ln(\text{contraste rec.}) / \ln(\text{contraste real})$ | Fracción del contraste recuperada en escala log. **El número más citable** |
| `z_centroide_*` | Profundidad del centro de la anomalía, solo dentro de la capa 2 y con el signo correcto | Revela si la inversión la pone más somera o más profunda |
| `rugosidad` | $\sqrt{\Phi_m}$: el mismo término de suavidad que minimiza la inversión (diferencias de $\ln\rho$ entre celdas vecinas) | Qué tan "estructurado" es el modelo. Eje x de la curva L |

Las medias de resistividad se calculan como **media geométrica**, no aritmética,
porque la resistividad es una magnitud que vive naturalmente en escala
logarítmica.

In [ ]:
def media_geom(v):
    """Media geometrica: la correcta para resistividades."""
    v = np.asarray(v)
    return float(np.exp(np.mean(np.log(v)))) if v.size else float('nan')


def rugosidad_modelo(mgr):
    """Rugosidad = raiz de Phi_m, el MISMO termino de suavidad que la
    inversion minimiza (Phi = Phi_d + lambda * Phi_m).

    Phi_m = || W_m (log m) ||^2, donde W_m es la matriz de restricciones
    de suavidad (diferencias de log-rho entre celdas vecinas).
    Se lo pedimos a la propia inversion en vez de recalcularlo a mano:
    la paraDomain de PyGIMLi NO guarda las fronteras entre celdas
    vecinas, asi que recorrer sus boundaries() daba siempre 0.
    """
    return float(np.sqrt(mgr.inv.phiModel()))


def malla_inversion():
    """Grilla regular de inversion, IDENTICA para todos los casos.

    DIMENSIONAMIENTO: hay 171 datos. Pedir muchas mas incognitas que
    datos, y sobre todo pedirlas a profundidades donde el arreglo no
    tiene sensibilidad, deja el problema tan mal condicionado que la
    inversion puede no encontrar ninguna mejora en la primera iteracion
    y devolver el modelo inicial.

    Por eso la malla llega a 6 m (la profundidad de investigacion de
    este arreglo es ~5-6 m) con 25 x 7 = 175 celdas, no a 10 m con 432.
    """
    dominio = pg.createGrid(
        x=np.linspace(-16, 16, 26),
        y=-pg.cat([0], pg.utils.grange(0.5, 6, n=7))[::-1],
        marker=2)
    # marco triangular grueso = "el resto del semiespacio", no se invierte
    return mt.appendTriangleBoundary(dominio, marker=1, xbound=50, ybound=50)


def revisar_inversion(modelPD, chi2, nombre):
    """Avisa FUERTE si la inversion no produjo un resultado utilizable.

    Sin esto, una inversion que se queda en el modelo inicial pasa
    desapercibida: devuelve numeros con pinta razonable (contraste 1.00,
    rugosidad 0.00) que uno podria meter al informe sin darse cuenta.
    """
    dispersion = float(np.std(np.log(modelPD)))
    problemas = []
    if dispersion < 1e-3:
        problemas.append(
            "el modelo invertido es HOMOGENEO (desviacion de log-rho = %.2e).\n"
            "        La inversion no itero: devolvio el modelo inicial." % dispersion)
    if chi2 > 50:
        problemas.append(
            "chi2 = %.0f es absurdamente alto; deberia acercarse a 1." % chi2)

    if problemas:
        print("\n  " + "!" * 62)
        print("  RESULTADO NO VALIDO en %s" % nombre)
        for p in problemas:
            print("      - " + p)
        print("  NO uses estos numeros. Revisa el log de iteraciones de arriba.")
        print("  " + "!" * 62 + "\n")
        return False
    return True

In [ ]:
def correr_caso(nombre, z1=-1.0, z2=-5.0,
                rho_capa1=100., rho_capa2=50., rho_basam=200., rho_obj=500.,
                geometria="circulo", lam=LAMBDA, mostrar=True):
    """
    Ciclo completo: modelo verdadero -> simulacion -> inversion -> metricas.

    z1, z2    : profundidades (negativas) de las dos interfaces
    rho_*     : resistividades en Ohm*m
    geometria : clave de GEOMETRIAS
    lam       : regularizador
    """
    print("=" * 68)
    print("CASO:", nombre)
    print("=" * 68)

    # --- 1. Modelo verdadero -----------------------------------------
    world = mt.createWorld(start=[-50, 0], end=[50, -50],
                           layers=[z1, z2], worldMarker=True)
    geom = world + crear_objeto(geometria)

    rhomap = [[M_CAPA1, rho_capa1],
              [M_CAPA2, rho_capa2],
              [M_BASAM, rho_basam],
              [M_OBJ,   rho_obj]]

    # --- 2. Esquema de medicion --------------------------------------
    scheme = ert.createData(elecs=np.linspace(X_ELEC[0], X_ELEC[1], N_ELEC),
                            schemeName=ESQUEMA)

    # refinamiento OBLIGATORIO en los electrodos (singularidad de la fuente)
    for p in scheme.sensors():
        geom.createNode(p)
        geom.createNode(p - [0, 0.1])

    mesh = mt.createMesh(geom, quality=QUALITY)

    # --- 3. Modelado directo + ruido ---------------------------------
    data = ert.simulate(mesh, scheme=scheme, res=rhomap,
                        noiseLevel=NOISE_LEVEL, noiseAbs=NOISE_ABS,
                        seed=SEED, verbose=False)

    n_total = data.size()
    # rhoa negativa es fisicamente imposible: viene de factores geometricos
    # enormes en las cuadruplas mas separadas, donde el voltaje es diminuto
    data.remove(data['rhoa'] < 0)
    n_malos = n_total - data.size()

    rhoa = np.array(data['rhoa'])
    print("  datos: %d  |  descartados (rhoa<0): %d" % (n_total, n_malos))
    print("  rhoa min/max: %.1f / %.1f Ohm*m" % (rhoa.min(), rhoa.max()))

    # --- 4. Inversion -------------------------------------------------
    mgr = ert.ERTManager(data)
    if MALLA_PROPIA:
        model = mgr.invert(data, mesh=malla_inversion(), lam=lam,
                           verbose=VERBOSE_INV)
    else:
        # el manager arma su propia malla a partir de los electrodos
        model = mgr.invert(data, lam=lam, verbose=VERBOSE_INV)
    modelPD = np.array(mgr.paraModel(model))

    chi2 = float(mgr.inv.chi2())
    try:
        rrms = float(mgr.inv.relrms())
    except TypeError:
        rrms = float(mgr.inv.relrms)
    print("  lambda = %-6g chi2 = %.3f   rRMS = %.2f %%" % (lam, chi2, rrms))

    # chequeo de sanidad ANTES de calcular metricas sobre basura
    valido = revisar_inversion(modelPD, chi2, nombre)

    # --- 5. Metricas de recuperacion ----------------------------------
    paraDomain = mgr.paraDomain

    # Para cada celda invertida, que region del modelo verdadero le toca.
    # Se resuelve analiticamente con marker_en(): sin mesh.findCell(),
    # sin scipy.sparse, y mucho mas rapido.
    marker_true = np.zeros(paraDomain.cellCount(), dtype=int)
    x_centro    = np.zeros(paraDomain.cellCount())
    z_centro    = np.zeros(paraDomain.cellCount())
    for i, c in enumerate(paraDomain.cells()):
        cx, cz = c.center()[0], c.center()[1]
        marker_true[i] = marker_en(cx, cz, z1, z2, geometria)
        x_centro[i], z_centro[i] = cx, cz

    mask_obj = marker_true == M_OBJ
    mask_enc = marker_true == M_CAPA2        # encajante del objeto

    rho_rec_obj = media_geom(modelPD[mask_obj])
    rho_rec_enc = media_geom(modelPD[mask_enc])

    contraste_real = rho_obj / rho_capa2
    contraste_rec  = rho_rec_obj / rho_rec_enc

    if abs(np.log(contraste_real)) > 1e-9:
        recuperacion = np.log(contraste_rec) / np.log(contraste_real)
    else:
        recuperacion = float('nan')

    print("  contraste  real: %7.2f   recuperado: %7.2f   (%.0f %% en log)"
          % (contraste_real, contraste_rec, 100 * recuperacion))

    # Centroide de la anomalia recuperada, ponderado por el apartamiento
    # respecto del encajante. Solo cuentan:
    #   - celdas dentro de la franja de la capa 2 (z2 < z < z1) y cerca
    #     del objeto (|x| <= 8 m): si no, el contraste de la capa 1 y del
    #     basamento tambien "pesa" y arrastra el centroide hacia ellos;
    #   - celdas que se apartan en el MISMO sentido que el objeto real
    #     (mas resistivas si el objeto es resistivo, y viceversa): asi el
    #     halo de signo contrario que crea la regularizacion no cuenta.
    dev  = np.log(modelPD / rho_rec_enc)
    peso = np.abs(dev)
    peso[np.sign(dev) != np.sign(np.log(contraste_real))] = 0.0
    peso[(z_centro > z1) | (z_centro < z2) | (np.abs(x_centro) > 8)] = 0.0
    peso[peso < 0.15] = 0.0
    z_rec = float(np.sum(peso * z_centro) / np.sum(peso)) if peso.sum() > 0 else float('nan')
    z_real = float(np.mean(z_centro[mask_obj])) if mask_obj.any() else float('nan')
    print("  centroide z  real: %.2f m   recuperado: %.2f m" % (z_real, z_rec))

    rug = rugosidad_modelo(mgr)
    print("  rugosidad del modelo: %.2f" % rug)

    # --- 6. Figura comparativa ----------------------------------------
    todas = [rho_capa1, rho_capa2, rho_basam, rho_obj]
    cmin, cmax = min(todas) * 0.8, max(todas) * 1.2

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6),
                                   sharex=True, sharey=True)
    pg.show(mesh, rhomap, ax=ax1, hold=True, cMap="Spectral_r", logScale=True,
            orientation="vertical", cMin=cmin, cMax=cmax, label=pg.unit('res'))
    ax1.set_title("Modelo verdadero  —  %s" % nombre)

    pg.show(paraDomain, modelPD, ax=ax2, hold=True, cMap="Spectral_r",
            logScale=True, orientation="vertical", cMin=cmin, cMax=cmax,
            label=pg.unit('res'))
    ax2.set_title(r"Inversion  ($\lambda$=%g, $\chi^2$=%.2f, rRMS=%.1f%%)"
                  % (lam, chi2, rrms))

    # contorno del objeto verdadero dibujado sobre la inversion:
    # asi se ve a simple vista cuanto se corrio, se achato o se esparcio
    o = OBJETOS[geometria]
    if o["tipo"] == "circulo":
        ax2.add_patch(plt.Circle(o["centro"], o["radio"], fill=False,
                                 ec="k", ls="--", lw=1.2))
    else:
        (x0, x1), (zz0, zz1) = sorted(o["x"]), sorted(o["z"])
        ax2.add_patch(plt.Rectangle((x0, zz0), x1 - x0, zz1 - zz0,
                                    fill=False, ec="k", ls="--", lw=1.2))
    for zi in (z1, z2):
        ax2.axhline(zi, c="k", ls=":", lw=0.8)

    for ax in (ax1, ax2):
        ax.set_xlim(-18, 18); ax.set_ylim(-10, 0); ax.set_ylabel("z (m)")
    ax2.set_xlabel("x (m)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "caso_%s.png" % nombre), dpi=150)
    if mostrar:
        plt.show()
    else:
        plt.close(fig)

    return dict(
        caso=nombre, valido=valido, geometria=geometria, lam=lam,
        z1=z1, z2=z2, espesor_capa2=abs(z2 - z1),
        rho_capa1=rho_capa1, rho_capa2=rho_capa2,
        rho_basam=rho_basam, rho_obj=rho_obj,
        n_datos=data.size(), n_descartados=n_malos,
        rhoa_min=round(float(rhoa.min()), 2),
        rhoa_max=round(float(rhoa.max()), 2),
        chi2=round(chi2, 3), rrms_pct=round(rrms, 2),
        rho_rec_objeto=round(rho_rec_obj, 1),
        rho_rec_encajante=round(rho_rec_enc, 1),
        contraste_real=round(contraste_real, 3),
        contraste_recuperado=round(contraste_rec, 3),
        recuperacion_log=round(float(recuperacion), 3),
        z_centroide_real=round(z_real, 2),
        z_centroide_recuperado=round(z_rec, 2),
        rugosidad=round(rug, 2),
    )

In [ ]:
# Caso base del que parten todos los experimentos:
#   capa 1:  0 a -1 m   100 Ohm*m
#   capa 2: -1 a -7 m    50 Ohm*m   <- aloja el objeto
#   basamento:          200 Ohm*m
#   objeto: circulo r=1.5 m en (0,-3), 500 Ohm*m (resistivo)
#
# El objeto queda SIEMPRE contenido dentro de la capa 2, en todos los
# experimentos. Si se saliera al basamento, el experimento A dejaria de
# medir el efecto del espesor y mediria una mezcla de dos cosas.
BASE = dict(z1=-1.0, z2=-7.0, rho_capa1=100., rho_capa2=50.,
            rho_basam=200., rho_obj=500., geometria="circulo", lam=20)

resultados = []   # se va llenando a medida que corren los experimentos

---
## Experimento A — Espesor de la capa que aloja el objeto

El objeto se queda quieto en $z = -3$ m (su base llega a $-4.5$ m). Lo que se
mueve es **el basamento**: espesores de capa 2 de 4, 6 y 10 m.

El objeto queda **contenido dentro de la capa 2 en los tres casos**. Esto es
deliberado: si en el caso delgado el objeto se saliera al basamento, estaríamos
midiendo a la vez el efecto del espesor y el de un objeto partido en dos
litologías, y no sabríamos a cuál atribuir la diferencia.

**Qué observar.** Con espaciamiento de 1.5 m y apertura de 30 m, la profundidad
de investigación del dipolo‑dipolo anda por los **5–6 m**.

- **A1 (4 m):** el basamento queda a solo 0.5 m bajo el objeto. ¿Se confunden las
  dos anomalías en una sola mancha?
- **A2 (6 m):** el contacto está justo en el límite de la resolución.
- **A3 (10 m):** el basamento cae fuera de alcance. ¿Qué pone la inversión ahí
  abajo — nada, o algo inventado?

Pregunta guía: *¿el método ve el contacto, o lo inventa donde se le acabó la
sensibilidad?*

> **Qué dio la corrida de referencia** (PyGIMLi 1.5.5, semilla 1337). Tus números
> pueden variar en el segundo decimal según la versión.
> - El objeto se recupera casi igual en los tres casos (37 %, 33 %, 33 % en log):
>   el espesor de la capa **no** cambia cuánto se ve el objeto.
> - Lo que cambia es el fondo. En A3 no hay basamento en los primeros 10 m, y aun
>   así la inversión pinta valores más altos hacia abajo. No es un basamento
>   inventado: donde los datos ya no tienen sensibilidad, el modelo **regresa al
>   modelo inicial** (la mediana de $\rho_a$, ~65 Ω·m). Esa zona refleja la
>   inversión, no el subsuelo.
> - Con el **mismo λ = 20**, A1 da χ² ≈ 2.1 y A3 χ² ≈ 0.96. Un λ fijo no garantiza
>   el mismo ajuste en modelos distintos: el λ "correcto" depende del subsuelo.

In [ ]:
for etiqueta, z2 in [("A1_espesor_4m",  -5.0),
                     ("A2_espesor_6m",  -7.0),
                     ("A3_espesor_10m", -11.0)]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, z2=z2)
    resultados.append(correr_caso(**cfg))

---
## Experimento B — Contraste de resistividad

El encajante se queda en 50 Ω·m. Solo cambia el objeto: contraste débil (×2),
muy resistivo (×20) y muy conductivo (÷10).

**Qué observar.** Hay una asimetría fundamental del ERT que debería aparecer aquí:
**los cuerpos conductivos se resuelven mejor que los resistivos**. La corriente se
concentra en lo conductivo (lo "busca") y rodea lo resistivo (lo "evita"), así que
un cuerpo resistivo deja menos huella en los datos. Compara B2 (resistivo ×20)
contra B3 (conductivo ÷10) usando `recuperacion_log`: ¿cuál recupera mayor
fracción de su contraste, pese a que el resistivo tiene contraste nominal más
fuerte? Y en B1, con contraste de apenas ×2, ¿la anomalía sobrevive al ruido?

> **Qué dio la corrida de referencia.**
> - B3 conductivo ÷10: **74 %** del contraste en log. B2 resistivo ×20: solo **29 %**.
>   Esto confirma la asimetría, aunque el resistivo tenga el doble de contraste nominal.
> - B1 (×2): se recupera 18 % y el contraste recuperado es 1.13, o sea, 13 % más
>   resistivo que el encajante. Con ruido del 1 % está al borde de lo detectable.
>   Su centroide (−4.4 m vs −2.8 m real) **no es confiable**: cuando la anomalía es
>   tan débil, el "centroide" se lo llevan las fluctuaciones del fondo.

In [ ]:
for etiqueta, rho_obj in [("B1_contraste_bajo_x2",  100.),
                          ("B2_resistivo_x20",     1000.),
                          ("B3_conductivo_div10",     5.)]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, rho_obj=rho_obj)
    resultados.append(correr_caso(**cfg))

---
## Experimento C — Geometría del objeto

Tres formas con área casi igual (≈ 7 m²): compacta, alargada vertical y alargada
horizontal.

**Qué observar.** Dos ideas compiten aquí, y conviene distinguirlas:

1. **La resolución cae con la profundidad.** Las partes someras de un cuerpo se
   ven mejor que las profundas. En un dique, el **techo** aparece y la **base**
   se pierde: no se puede decir hasta dónde baja.
2. **El dipolo‑dipolo es muy sensible a los cambios laterales.** Por eso detecta
   bien un cuerpo vertical angosto y lo ubica bien en x. Es el arreglo preferido
   para diques, fallas y cavidades. Wenner, en cambio, favorece las capas
   horizontales.

Y un tercer concepto que este experimento muestra muy bien: el **principio de
equivalencia**. Una capa **delgada y resistiva** dentro de un medio conductivo se
"ve" solo a través de su resistencia transversal $T = \rho \cdot h$. La inversión
no puede separar ρ y h: los reparte en una capa más gruesa y menos resistiva.

Mira también los **artefactos**: suele aparecer un halo de signo contrario
alrededor de la anomalía (aquí, zonas más conductivas que el encajante a los
lados). No existen en el subsuelo: los produce la regularización.

> **Qué dio la corrida de referencia.**
> - Dique vertical: **40 %**, el mejor de los tres. Sale bien centrado en x, pero
>   solo se ve su mitad superior. La base (−5.3 m) no aparece.
> - Círculo: **33 %**.
> - Tabular horizontal (1 m de espesor): **26 %**, el peor. Sale como una mancha
>   difusa y más profunda (centroide −3.8 m vs −3.1 m real). Es el principio de
>   equivalencia en acción.
>
> Si esperabas que el tabular saliera mejor, este es justamente el aprendizaje:
> "horizontal" no significa "fácil" cuando el cuerpo es delgado.

In [ ]:
for etiqueta, geo in [("C1_circulo",     "circulo"),
                      ("C2_dique_vert",  "dique_vert"),
                      ("C3_tabular_hor", "tabular_hor")]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, geometria=geo)
    resultados.append(correr_caso(**cfg))

---
## Experimento D — El regularizador λ

Este es el experimento que casi siempre cae en la sustentación, porque λ es el
único parámetro de la inversión que el usuario elige **a dedo** y que no viene de
los datos.

Recordando el funcional que se minimiza:

$$\Phi = \Phi_d + \lambda\,\Phi_m$$

- **λ pequeño** → el término de suavidad casi no pesa. La inversión persigue los
  datos hasta el último decimal, incluido el ruido. Resultado: modelo rugoso,
  lleno de estructura espuria, $\chi^2 < 1$ (sobreajuste).
- **λ grande** → domina la suavidad. Modelo borroso, con poca estructura, que ni
  siquiera ajusta los datos: $\chi^2 \gg 1$ (subajuste).
- **λ adecuado** → $\chi^2 \approx 1$: se ajustan los datos exactamente hasta su
  nivel de ruido y ni un poco más.

Usamos **cinco** valores en vez de tres porque con cinco puntos la curva L se ve
de verdad. La **curva L** grafica el desajuste contra la rugosidad
($\sqrt{\Phi_m}$). El λ óptimo está en el codo, donde se pasa de "ganar mucho
ajuste por poca estructura" a "ganar poco ajuste a costa de mucha estructura".

> **Qué dio la corrida de referencia.**
>
> | λ | χ² | rRMS | rugosidad | contraste recuperado |
> |--:|--:|--:|--:|--:|
> | 1 | 0.55 | 0.75 % | 5.6 | 57 % |
> | 5 | 0.60 | 0.78 % | 4.4 | 45 % |
> | **20** | **1.06** | **1.04 %** | **3.8** | **33 %** |
> | 100 | 2.19 | 1.49 % | 3.2 | 22 % |
> | 500 | 7.15 | 2.68 % | 2.6 | 15 % |
>
> **Ojo con la trampa:** λ = 1 recupera *más* contraste (57 %) que λ = 20, pero
> tiene χ² = 0.55: le está creyendo al ruido. Ese contraste extra no se puede
> defender, porque parte de la estructura que lo produce es espuria. El criterio
> para elegir λ es **χ² ≈ 1** (aquí, λ = 20), no "el que más se parezca a lo que
> esperaba". En campo no conoces el modelo verdadero: solo tienes χ².

In [ ]:
resultados_lam = []
for lam in [1, 5, 20, 100, 500]:
    cfg = dict(BASE)
    cfg.update(nombre="D_lambda_%g" % lam, lam=lam)
    r = correr_caso(**cfg)
    resultados_lam.append(r)
    resultados.append(r)

In [ ]:
# ------------------------------------------------------------------
# CURVA L  y  evolucion de chi2
# ------------------------------------------------------------------
dl = pd.DataFrame(resultados_lam)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Curva L: desajuste vs rugosidad ---
ax1.plot(dl["rugosidad"], dl["rrms_pct"], 'o-', color="#2d6a9f",
         lw=1.8, ms=8, zorder=2)
for _, r in dl.iterrows():
    ax1.annotate(r"$\lambda$=%g" % r["lam"],
                 (r["rugosidad"], r["rrms_pct"]),
                 textcoords="offset points", xytext=(9, 6), fontsize=10)
ax1.set_xlabel("Rugosidad del modelo  (estructura)")
ax1.set_ylabel("rRMS (%)   —  desajuste")
ax1.set_title("Curva L\nel codo marca el compromiso optimo")
ax1.grid(alpha=0.3)

# --- chi2 vs lambda ---
ax2.semilogx(dl["lam"], dl["chi2"], 'o-', color="#b5523a", lw=1.8, ms=8)
ax2.axhline(1.0, ls="--", c="gray", lw=1.2)
ax2.annotate(r"$\chi^2=1$: ajuste ideal", (dl["lam"].min(), 1.0),
             textcoords="offset points", xytext=(6, 8),
             fontsize=10, color="gray")
ax2.set_xlabel(r"$\lambda$")
ax2.set_ylabel(r"$\chi^2$")
ax2.set_title(r"Ajuste a los datos vs regularizacion")
ax2.grid(alpha=0.3, which="both")

fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "curva_L.png"), dpi=150)
plt.show()

print("\nlambda que deja chi2 mas cerca de 1: %g"
      % dl.loc[(dl["chi2"] - 1).abs().idxmin(), "lam"])

---
## 5. Tabla resumen

Todas las métricas de los 14 casos, exportadas a CSV para que las lleves al
informe.

In [ ]:
df = pd.DataFrame(resultados)
df.to_csv(os.path.join(OUTDIR, "metricas.csv"), index=False)

n_malos = int((~df["valido"]).sum())
if n_malos:
    print("!!! %d de %d casos NO son validos: %s"
          % (n_malos, len(df), list(df.loc[~df["valido"], "caso"])))
    print("!!! No los uses en el informe sin revisar por que fallaron.")
else:
    print("Los %d casos dieron resultados utilizables." % len(df))

cols = ["caso", "valido", "chi2", "rrms_pct", "contraste_real",
        "contraste_recuperado", "recuperacion_log", "z_centroide_real",
        "z_centroide_recuperado", "rugosidad", "n_descartados"]
df[cols]

In [ ]:
# ------------------------------------------------------------------
# Figura resumen: cuanto contraste recupera cada caso
# ------------------------------------------------------------------
sub = df[(~df["caso"].str.startswith("D_")) & df["valido"]]

fig, ax = plt.subplots(figsize=(9, 4.5))
colores = ["#2d6a9f" if c.startswith("A") else
           "#b5523a" if c.startswith("B") else
           "#4f8c5a" for c in sub["caso"]]
ax.bar(range(len(sub)), sub["recuperacion_log"] * 100, color=colores)
ax.set_xticks(range(len(sub)))
ax.set_xticklabels(sub["caso"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Contraste recuperado (% en escala log)")
ax.set_title("Fidelidad de la inversion por caso\n(azul: espesor · rojo: contraste · verde: geometria)")
ax.axhline(100, ls="--", c="gray", lw=1.2)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "resumen_recuperacion.png"), dpi=150)
plt.show()

---
# Punto 3 — Tres geometrías: circular, triangular y esférica

**Qué se pide.** Tres casos con geometrías distintas y, para cada uno, evaluar:

1. el **número de electrodos**,
2. el **regularizador λ**,
3. la **malla** (el tamaño de los triángulos), tanto la de **simulación** como la de **inversión**.

### Un medio homogéneo, para aislar la forma

Aquí el objeto (500 Ω·m) está en un **medio homogéneo de 50 Ω·m**, sin capas. Hay
dos razones:

- **Comparación justa.** Si hubiera capas, parte de la diferencia entre casos
  vendría de ellas. Sin capas, lo único que cambia entre los tres casos es la forma.
- **La esfera necesita 3D**, y PyGIMLi no permite capas en mundos 3D
  (`createWorld` responde *"3D with layers is not yet implemented"*).

### ¿Por qué "círculo" y "esfera" no son lo mismo?

La ERT 2D en realidad es **2.5D**: supone que el subsuelo **no cambia en la
dirección perpendicular a la línea** de electrodos. Por eso un círculo en la
sección 2D es en realidad un **cilindro infinito** que atraviesa la línea.

| Caso | Qué es en la realidad | Cómo se simula |
|---|---|---|
| **Circular** | Cilindro infinito (tubo, túnel, conducto) | 2.5D, malla de triángulos |
| **Triangular** | Prisma infinito de sección triangular (cuña) | 2.5D, malla de triángulos |
| **Esférico** | Bola de verdad (bloque, cavidad aislada) | **3D**, malla de tetraedros |

Los tres tienen la **misma sección en el plano de la línea** (≈ 7 m², centro a
−3 m). Los datos de la esfera se simulan en 3D pero **se invierten en 2D**, igual
que en campo: en la realidad los cuerpos son 3D y casi siempre se interpretan con
una inversión 2D.

> **Detalle técnico de la esfera.** En 3D, PyGIMLi no da el factor geométrico
> correcto si se deja la eliminación de singularidad (`sr`) activada: un medio
> homogéneo de 50 Ω·m daba ρa = 25. Por eso se calcula el **factor geométrico
> numérico**, simulando un medio homogéneo de 1 Ω·m en la misma malla ($k = 1/u$),
> que es como lo hace la propia función `createGeometricFactors` de PyGIMLi. Así,
> un medio homogéneo da exactamente 50 Ω·m y el error de la malla se cancela.
> Para mallar en 3D hace falta el programa **TetGen** (`conda install -c conda-forge tetgen`,
> ya incluido en `instalar_entorno.bat`).

### Diseño: una variable a la vez

| Subpunto | Lo que cambia | Valores | Lo demás queda en |
|---|---|---|---|
| 3.1 | Electrodos (misma línea de 30 m) | 11, **21**, 41 | λ=20, mallas medias |
| 3.2 | λ | 2, **20**, 200 | 21 electrodos, mallas medias |
| 3.3a | Malla de **simulación** (directa) | gruesa, **media**, fina | 21 electrodos, λ=20 |
| 3.3b | Malla de **inversión** | gruesa, **media**, fina | 21 electrodos, λ=20 |

En negrita está el **caso base**, que se repite en los cuatro subpuntos como
referencia.

**Métricas nuevas, pensadas para leerse fácil:**

| Métrica | Qué dice |
|---|---|
| `rho_max_rec` | La resistividad más alta que encontró la inversión cerca del objeto. El objeto real tiene **500**. |
| `recuperacion_log` | Qué fracción del contraste ×10 se recuperó (en escala log), igual que en los experimentos A–D. |
| `anomalia_rhoa_pct` | Cuánto se aparta el ρa medido del fondo (50 Ω·m), en %. Es lo que "ve" el equipo **antes** de invertir. |
| `error_malla_pct` | Error numérico de la malla de simulación: en un medio homogéneo, ρa debería dar exacto. |
| `n_celdas_inv` | Número de incógnitas de la inversión. |

In [ ]:
# ==================================================================
# PUNTO 3 - configuracion y funciones
# ==================================================================
import time
from matplotlib.path import Path
from matplotlib.colors import LogNorm

OUT_P3 = os.path.join(OUTDIR, "punto3")
os.makedirs(OUT_P3, exist_ok=True)

RHO_FONDO = 50.     # medio homogeneo
RHO_OBJ3  = 500.    # objeto resistivo (contraste x10)
M_FONDO3, M_OBJ3 = 1, 2

# --- Las tres geometrias: misma seccion (~7 m2) y centro a ~-3 m ---------
GEO_P3 = {
    # cilindro infinito perpendicular a la linea (2.5D)
    "circular":   dict(dim=2, tipo="circulo", centro=(0.0, -3.0), radio=1.5),
    # prisma infinito de seccion triangular, punta hacia abajo
    # base 4 m, altura 3.5 m -> area 7.0 m2, centroide en z = -2.97 m
    "triangular": dict(dim=2, tipo="poligono",
                       vertices=[(-2.0, -1.8), (2.0, -1.8), (0.0, -5.3)]),
    # esfera 3D; su corte en el plano de la linea es el mismo circulo
    "esferico":   dict(dim=3, tipo="circulo", centro=(0.0, -3.0), radio=1.5),
}
NOMBRES_GEO = list(GEO_P3.keys())

# --- Variantes de la malla de SIMULACION (directa) ------------------------
#  2D: quality = angulo minimo de los triangulos; area_obj = area maxima (m2)
#      de los triangulos dentro del objeto; area = area maxima en todo el mundo
#  3D: quality = calidad de los tetraedros; area_obj = area maxima (m2) de los
#      triangulos de la superficie de la esfera
MALLAS_DIR = {
    "gruesa": dict(q2=30, area_obj2=0.5,  area2=0.0, q3=1.5, area_obj3=1.0),
    "media":  dict(q2=34, area_obj2=0.05, area2=0.0, q3=1.3, area_obj3=0.3),
    "fina":   dict(q2=34, area_obj2=0.01, area2=1.0, q3=1.2, area_obj3=0.1),
}

# --- Variantes de la malla de INVERSION -----------------------------------
#  paraDX: separacion de nodos en superficie, relativa al espaciamiento
#          entre electrodos (1 = solo los electrodos; 0.2 = 4 nodos extra)
#  paraMaxCellSize: area maxima (m2) de cada celda de inversion
MALLAS_INV = {
    "gruesa": dict(paraDX=1.0,  paraMaxCellSize=10.0),
    "media":  dict(paraDX=0.33, paraMaxCellSize=1.0),
    "fina":   dict(paraDX=0.2,  paraMaxCellSize=0.25),
}
PARA_DEPTH = 8.0     # profundidad de la malla de inversion, fija para todos

BASE3 = dict(n_elec=21, lam=20, malla_dir="media", malla_inv="media")


def esquema_p3(n_elec):
    """Arreglo dipolo-dipolo de n_elec electrodos sobre la MISMA linea."""
    return ert.createData(elecs=np.linspace(X_ELEC[0], X_ELEC[1], n_elec),
                          schemeName=ESQUEMA)


def objeto_plc(geo, malla):
    """Poligono (2D) o esfera (3D) del objeto, listo para la malla."""
    g, m = GEO_P3[geo], MALLAS_DIR[malla]
    if g["dim"] == 3:
        cx, cz = g["centro"]
        # OJO: createSphere NO tiene parametro "radius" (lo ignora en
        # silencio y deja una esfera de 1 m de DIAMETRO). Se usa size =
        # diametro en x, y, z.
        d = 2 * g["radio"]
        return mt.createSphere(pos=[cx, 0.0, cz], size=[d, d, d],
                               nSegments=32, nRings=16,
                               marker=M_OBJ3, boundaryMarker=10,
                               area=m["area_obj3"])
    if g["tipo"] == "circulo":
        return mt.createCircle(pos=list(g["centro"]), radius=g["radio"],
                               nSegments=48, marker=M_OBJ3,
                               boundaryMarker=10, area=m["area_obj2"])
    return mt.createPolygon(g["vertices"], isClosed=True, marker=M_OBJ3,
                            boundaryMarker=10, area=m["area_obj2"])


def dentro_p3(x, z, geo):
    """Esta el punto (x, z) del plano de la linea dentro del objeto?"""
    g = GEO_P3[geo]
    if g["tipo"] == "circulo":
        cx, cz = g["centro"]
        return (x - cx) ** 2 + (z - cz) ** 2 <= g["radio"] ** 2
    return Path(g["vertices"]).contains_point((x, z))


_cache_datos = {}

def simular_p3(geo, n_elec, malla_dir):
    """Malla verdadera + datos sinteticos con ruido. Se guarda en cache:
    variar lambda o la malla de inversion NO requiere volver a simular."""
    clave = (geo, n_elec, malla_dir)
    if clave in _cache_datos:
        return _cache_datos[clave]

    dim, m = GEO_P3[geo]["dim"], MALLAS_DIR[malla_dir]
    scheme = esquema_p3(n_elec)

    if dim == 2:
        world = mt.createWorld(start=[-50, 0], end=[50, -50], worldMarker=True)
        geom = world + objeto_plc(geo, malla_dir)
        for p in scheme.sensors():
            geom.createNode(p)
            geom.createNode(p - [0, 0.1])    # refinamiento en el electrodo
        mesh = mt.createMesh(geom, quality=m["q2"], area=m["area2"])
        sr = True
    else:
        world = mt.createWorld(start=[-50, -50, 0], end=[50, 50, -50],
                               worldMarker=True)
        geom = world + objeto_plc(geo, malla_dir)
        for p in scheme.sensors():
            geom.createNode(p)
            geom.createNode(p - [0, 0, 0.1])  # refinamiento en el electrodo
        mesh = mt.createMesh(geom, quality=m["q3"])
        sr = False
        # factor geometrico NUMERICO (k = 1/u en un medio de 1 Ohm*m)
        hom = ert.simulate(mesh, scheme=scheme, res=1.0, sr=False,
                           calcOnly=True, verbose=False)
        scheme["k"] = 1.0 / hom["u"]

    # error de la malla: medio homogeneo con el factor geometrico ANALITICO.
    # Si la malla fuera perfecta, rhoa = 1 exacto en todas las cuadruplas.
    s_an = pg.DataContainerERT(scheme)
    s_an["k"] = pg.core.geometricFactors(s_an, dim=3)
    # OJO: guardar el contenedor en una variable ANTES de sacar ["rhoa"].
    # simulate(...)["rhoa"] devuelve una referencia a memoria del contenedor
    # temporal, que Python libera enseguida -> datos basura o crash.
    d_hom = ert.simulate(mesh, scheme=s_an, res=1.0, sr=sr,
                         noiseLevel=0, verbose=False)
    rh_hom = np.array(d_hom["rhoa"])
    error_malla = 100 * float(np.max(np.abs(rh_hom - 1.0)))

    rhomap = [[M_FONDO3, RHO_FONDO], [M_OBJ3, RHO_OBJ3]]
    limpio = ert.simulate(mesh, scheme=scheme, res=rhomap, sr=sr,
                          noiseLevel=0, verbose=False)
    data = ert.simulate(mesh, scheme=scheme, res=rhomap, sr=sr,
                        noiseLevel=NOISE_LEVEL, noiseAbs=NOISE_ABS,
                        seed=SEED, verbose=False)
    data.remove(data["rhoa"] < 0)

    anomalia = 100 * float(np.max(np.abs(np.array(limpio["rhoa"]) / RHO_FONDO - 1)))
    res = dict(data=data, n_celdas_dir=mesh.cellCount(),
               error_malla_pct=round(error_malla, 2),
               anomalia_rhoa_pct=round(anomalia, 1))
    _cache_datos[clave] = res
    return res


def invertir_p3(geo, subpunto, variante, n_elec=21, lam=20,
                malla_dir="media", malla_inv="media"):
    """Simula (o reusa) los datos, invierte en 2D y calcula metricas."""
    t0 = time.time()
    sim = simular_p3(geo, n_elec, malla_dir)
    data = pg.DataContainerERT(sim["data"])
    # la inversion siempre es 2D: electrodos en (x, 0)
    for i in range(data.sensorCount()):
        data.setSensorPosition(i, [data.sensorPosition(i)[0], 0.0, 0.0])
    # la inversion 2D usa el factor geometrico ANALITICO de semiespacio.
    # (rhoa ya viene corregido; el k numerico solo servia para la malla 3D)
    data["k"] = pg.core.geometricFactors(data, dim=3)

    imesh = ert.createInversionMesh(data, quality=33, paraDepth=PARA_DEPTH,
                                    **MALLAS_INV[malla_inv])
    mgr = ert.ERTManager(data)
    model = mgr.invert(data, mesh=imesh, lam=lam, verbose=False)
    modelPD = np.array(mgr.paraModel(model))
    pd_ = mgr.paraDomain
    chi2 = float(mgr.inv.chi2())
    valido = revisar_inversion(modelPD, chi2, "%s / %s" % (geo, variante))

    xc = np.array([c.center()[0] for c in pd_.cells()])
    zc = np.array([c.center()[1] for c in pd_.cells()])
    en_obj = np.array([dentro_p3(x, z, geo) for x, z in zip(xc, zc)])
    cerca = (np.abs(xc) <= 10) & (zc >= -7)
    rho_obj = media_geom(modelPD[en_obj])
    rho_fon = media_geom(modelPD[cerca & ~en_obj])
    rec = np.log(rho_obj / rho_fon) / np.log(RHO_OBJ3 / RHO_FONDO)
    rho_max = float(modelPD[(np.abs(xc) <= 5) & (zc >= -7)].max())

    r = dict(geometria=geo, subpunto=subpunto, variante=variante,
             n_elec=n_elec, lam=lam, malla_dir=malla_dir, malla_inv=malla_inv,
             valido=valido, n_datos=data.size(),
             n_celdas_dir=sim["n_celdas_dir"], n_celdas_inv=pd_.cellCount(),
             error_malla_pct=sim["error_malla_pct"],
             anomalia_rhoa_pct=sim["anomalia_rhoa_pct"],
             chi2=round(chi2, 2), rrms_pct=round(float(mgr.inv.relrms()), 2),
             rho_max_rec=round(rho_max, 1),
             recuperacion_log=round(float(rec), 3),
             rugosidad=round(rugosidad_modelo(mgr), 2),
             tiempo_s=round(time.time() - t0, 1),
             _pd=pd_, _model=modelPD)
    print("  %-10s %-14s datos=%4d celdas_inv=%5d  chi2=%5.2f  "
          "rho_max=%6.1f  recup=%3.0f%%"
          % (geo, variante, r["n_datos"], r["n_celdas_inv"], r["chi2"],
             r["rho_max_rec"], 100 * r["recuperacion_log"]))
    return r


def contorno_p3(ax, geo, **kw):
    """Dibuja el contorno verdadero del objeto (corte en el plano de la linea)."""
    g = GEO_P3[geo]
    estilo = dict(fill=False, ec="k", ls="--", lw=1.2)
    estilo.update(kw)
    if g["tipo"] == "circulo":
        ax.add_patch(plt.Circle(g["centro"], g["radio"], **estilo))
    else:
        ax.add_patch(plt.Polygon(g["vertices"], closed=True, **estilo))


NORM_P3 = LogNorm(vmin=30, vmax=600)

def figura_p3(resultados_sub, titulo, archivo):
    """Grilla 3x3: filas = geometrias, columnas = variantes."""
    variantes = list(dict.fromkeys(r["variante"] for r in resultados_sub))
    fig, axes = plt.subplots(len(NOMBRES_GEO), len(variantes),
                             figsize=(4.2 * len(variantes), 2.6 * len(NOMBRES_GEO)),
                             sharex=True, sharey=True, squeeze=False)
    for i, geo in enumerate(NOMBRES_GEO):
        for j, var in enumerate(variantes):
            ax = axes[i, j]
            r = next(x for x in resultados_sub
                     if x["geometria"] == geo and x["variante"] == var)
            pg.show(r["_pd"], r["_model"], ax=ax, hold=True, colorBar=False,
                    cMap="Spectral_r", logScale=True, cMin=30, cMax=600)
            contorno_p3(ax, geo)
            ax.set_xlim(-12, 12); ax.set_ylim(-8, 0)
            ax.set_title("%s | %s\nchi2=%.2f  rho_max=%.0f"
                         % (geo, var, r["chi2"], r["rho_max_rec"]), fontsize=9)
            ax.set_xlabel(""); ax.set_ylabel("")
    for ax in axes[-1]:
        ax.set_xlabel("x (m)")
    for ax in axes[:, 0]:
        ax.set_ylabel("z (m)")
    sm = plt.cm.ScalarMappable(norm=NORM_P3, cmap="Spectral_r")
    cb = fig.colorbar(sm, ax=axes, shrink=0.8, label=r"Resistividad ($\Omega$m)")
    cb.set_ticks([30, 50, 100, 200, 500])
    cb.set_ticklabels(["30", "50", "100", "200", "500"])
    fig.suptitle(titulo, fontsize=12)
    fig.savefig(os.path.join(OUT_P3, archivo), dpi=150, bbox_inches="tight")
    plt.show()


resultados_p3 = []
print("Punto 3 listo. Resultados en:", OUT_P3)

In [ ]:
# ------------------------------------------------------------------
# Los tres modelos verdaderos (corte en el plano de la linea) y cuanto
# se nota cada uno en los DATOS, antes de invertir
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), sharey=True)
x_e = np.linspace(X_ELEC[0], X_ELEC[1], BASE3["n_elec"])
for ax, geo in zip(axes, NOMBRES_GEO):
    ax.set_facecolor(plt.cm.Spectral_r(NORM_P3(RHO_FONDO)))
    contorno_p3(ax, geo, fill=True, fc=plt.cm.Spectral_r(NORM_P3(RHO_OBJ3)),
                ls="-", ec="k")
    ax.plot(x_e, np.zeros_like(x_e), "kv", ms=5, clip_on=False)
    ax.set_xlim(-16, 16); ax.set_ylim(-8, 0); ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    extra = ("\n(3D: una bola; esto es su corte)" if GEO_P3[geo]["dim"] == 3
             else "\n(2.5D: se extiende infinito hacia y)")
    ax.set_title(geo + extra, fontsize=10)
axes[0].set_ylabel("z (m)")
fig.suptitle("Punto 3 - modelos verdaderos: 500 Ohm*m dentro de 50 Ohm*m", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_P3, "p3_modelos_verdaderos.png"), dpi=150)
plt.show()

print("Anomalia maxima en rhoa (sin ruido), con el caso base:")
for geo in NOMBRES_GEO:
    s = simular_p3(geo, BASE3["n_elec"], BASE3["malla_dir"])
    print("  %-10s  %5.1f %%   (el ruido es del %g %%)"
          % (geo, s["anomalia_rhoa_pct"], NOISE_LEVEL))

---
## 3.1 — Número de electrodos

La línea mide siempre **30 m**. Lo que cambia es cuántos electrodos hay en ella,
y por lo tanto el espaciamiento:

| Electrodos | Espaciamiento | Qué esperar |
|---|---|---|
| 11 | 3.0 m | Pocos datos y celdas grandes cerca de la superficie: el objeto (de 3 m de ancho) queda "entre dos electrodos" |
| 21 | 1.5 m | Caso base |
| 41 | 0.75 m | Muchos más datos y mejor resolución somera, pero la **profundidad de investigación no aumenta**, porque la apertura total sigue siendo de 30 m |

Pregunta guía: *¿más electrodos siempre es mejor? ¿Qué gano y qué no gano?*

> **Qué dio la corrida de referencia** (contraste recuperado en log; entre paréntesis, la ρ máxima recuperada, con el objeto real en 500 Ω·m).
>
> | | 11 electrodos (36 datos) | 21 (171 datos) | 41 (741 datos) |
> |---|--:|--:|--:|
> | Circular | 32 % (126) | 45 % (200) | 53 % (297) |
> | Triangular | 34 % (136) | 45 % (214) | 55 % (313) |
> | Esférico | 8 % (64) | 12 % (80) | 13 % (86) |
>
> - Más electrodos **sí mejoran** la imagen: más datos y más finos cerca de la superficie, así que el objeto sale más contrastado y mejor delimitado.
> - La ganancia se reduce: de 11 a 21 electrodos se gana mucho más que de 21 a 41. Y la profundidad de investigación **no cambia**, porque la línea sigue midiendo 30 m.
> - Con 11 electrodos (cada 3 m) el triángulo ya no se distingue de un círculo: la forma necesita un espaciamiento menor que el tamaño del objeto.

In [ ]:
sub_elec = []
for geo in NOMBRES_GEO:
    for n in [11, 21, 41]:
        r = invertir_p3(geo, "3.1 electrodos", "%d electrodos" % n, n_elec=n)
        sub_elec.append(r)
resultados_p3 += sub_elec
figura_p3(sub_elec, "3.1 - Numero de electrodos (linea de 30 m, lambda=20)",
          "p3_1_electrodos.png")

---
## 3.2 — El regularizador λ

Igual que en el Experimento D, pero ahora con las tres geometrías:

- **λ = 2**: le cree mucho a los datos. El objeto sale más contrastado, pero con χ² < 1 (sobreajuste: también "ajusta" el ruido).
- **λ = 20**: caso base.
- **λ = 200**: muy suave. El objeto se "borra" y χ² sube por encima de 1.

Pregunta guía: *¿el λ que conviene es el mismo para las tres geometrías?* Fíjate
sobre todo en la esfera: su anomalía en los datos es tan débil que un λ grande
puede borrarla por completo.

> **Qué dio la corrida de referencia.**
>
> | | λ = 2 | λ = 20 | λ = 200 |
> |---|--:|--:|--:|
> | Circular | 57 % (χ² 0.54) | 45 % (χ² 1.00) | 33 % (χ² 2.02) |
> | Triangular | 58 % (χ² 0.56) | 45 % (χ² 0.80) | 35 % (χ² 2.00) |
> | Esférico | 16 % (χ² 0.51) | 12 % (χ² 0.63) | 9 % (χ² 0.83) |
>
> - Mismo patrón que en el Experimento D: λ chico da más contraste, pero con χ² < 1 (le cree al ruido); λ grande suaviza y χ² sube.
> - **La esfera es distinta:** su χ² queda por debajo de 1 **incluso con λ = 200**. Su señal es tan débil (10 % en ρa) que un modelo casi liso ya la explica. Si se eligiera λ con el criterio χ² ≈ 1, la esfera pediría un λ todavía mayor, y la anomalía prácticamente desaparecería. Un objeto chico en 3D puede quedar **por debajo de lo que la inversión 2D está dispuesta a mostrar**.

In [ ]:
sub_lam = []
for geo in NOMBRES_GEO:
    for lam in [2, 20, 200]:
        r = invertir_p3(geo, "3.2 lambda", "lambda=%g" % lam, lam=lam)
        sub_lam.append(r)
resultados_p3 += sub_lam
figura_p3(sub_lam, "3.2 - Regularizador lambda (21 electrodos)",
          "p3_2_lambda.png")

---
## 3.3 — La malla: tamaño de los triángulos

En ERT hay **dos mallas distintas** y conviene no confundirlas:

| | Malla de **simulación** (directa) | Malla de **inversión** |
|---|---|---|
| Para qué sirve | Calcular los datos sintéticos (el "subsuelo verdadero") | Representar el modelo que se busca: cada celda es una incógnita |
| Qué pasa si es gruesa | Los datos tienen **error numérico** (no son exactos) | La imagen sale "pixelada" y no puede representar bordes finos |
| Qué pasa si es fina | Datos más exactos, pero más lenta | Más incógnitas: el problema queda más indeterminado y depende más de λ |
| Parámetro en PyGIMLi | `quality`, `area` en `mt.createMesh` | `paraDX`, `paraMaxCellSize` en `ert.createInversionMesh` |

**Una regla práctica.** La malla de simulación tiene que ser lo bastante fina para
que su error numérico quede **por debajo del ruido de los datos (1 %)**. Afinar la
malla de inversión, en cambio, **no agrega información**: solo reparte la misma
información de los datos entre más celdas.

### 3.3a — Malla de simulación

`error_malla_pct` mide cuánto se equivoca la malla en un medio homogéneo, donde
ρa debería salir exacto. En la esfera (3D) ese error se cancela al usar el factor
geométrico numérico; aquí se reporta igual para que veas qué tan exacta es cada
malla 3D.

> **Qué dio la corrida de referencia** (la tabla completa se imprime debajo de la figura).
>
> - En 2D el error en medio homogéneo es **0.3 % en las tres mallas**, por debajo del ruido del 1 %. Ese error lo dominan los nodos refinados junto a los electrodos, que están en las tres.
> - Lo que sí cambia es **qué tan bien la malla dibuja el objeto**. El triángulo, con esquinas agudas, da una anomalía de 42.9 % en ρa con la malla gruesa, contra 47.9 % con la media y 49.0 % con la fina. El círculo casi no cambia (35.5 → 35.8 %). Una malla gruesa "redondea" las esquinas del objeto y cambia los datos.
> - En 3D el error crudo es grande (17–19 %), pero se cancela con el factor geométrico numérico. Como verificación aparte se modeló en 3D un prisma muy largo: dio 43 % de anomalía, contra 49 % del mismo prisma en 2D, así que el cálculo 3D es confiable a ~10 %.
> - En la inversión final la malla de simulación casi no influye (43–46 %): **lo importante es que su error quede por debajo del ruido**, y aquí se cumple.

In [ ]:
sub_mdir = []
for geo in NOMBRES_GEO:
    for malla in ["gruesa", "media", "fina"]:
        r = invertir_p3(geo, "3.3a malla simulacion", "simulacion " + malla,
                        malla_dir=malla)
        sub_mdir.append(r)
resultados_p3 += sub_mdir
figura_p3(sub_mdir, "3.3a - Malla de SIMULACION (21 electrodos, lambda=20)",
          "p3_3a_malla_simulacion.png")

print("\nTamano y exactitud de cada malla de simulacion:")
print(pd.DataFrame(sub_mdir)[["geometria", "variante", "n_celdas_dir",
                              "error_malla_pct", "anomalia_rhoa_pct"]]
      .to_string(index=False))

### 3.3b — Malla de inversión

Se usan los **mismos datos** (no se vuelve a simular). Solo cambia en cuántas
celdas se reparte el modelo:

| Malla | `paraDX` | `paraMaxCellSize` | Idea |
|---|---|---|---|
| gruesa | 1.0 | 10 m² | Un nodo por electrodo y celdas grandes |
| media | 0.33 | 1 m² | Caso base |
| fina | 0.2 | 0.25 m² | 4 nodos extra entre electrodos y celdas pequeñas |

> **Qué dio la corrida de referencia.**
>
> | | gruesa (293 celdas) | media (726) | fina (2069) |
> |---|--:|--:|--:|
> | Circular | 48 % | 45 % | 45 % |
> | Triangular | 46 % | 45 % | 46 % |
> | Esférico | 14 % | 12 % | 12 % |
>
> - **Afinar la malla de inversión no mejora la imagen.** Con 7 veces más celdas el resultado es el mismo: los 171 datos son la información disponible, y repartirlos en más celdas no crea información nueva. La regularización decide cómo se rellenan las celdas extra.
> - La malla gruesa recupera un poco más porque sus celdas grandes "promedian" el objeto, pero la imagen sale pixelada.
> - Moraleja práctica: las celdas de inversión deben ser de **un tercio a la mitad del espaciamiento** cerca de la superficie, que es lo que da la malla media (`paraDX = 0.33`). Más fino solo cuesta tiempo.

In [ ]:
sub_minv = []
for geo in NOMBRES_GEO:
    for malla in ["gruesa", "media", "fina"]:
        r = invertir_p3(geo, "3.3b malla inversion", "inversion " + malla,
                        malla_inv=malla)
        sub_minv.append(r)
resultados_p3 += sub_minv
figura_p3(sub_minv, "3.3b - Malla de INVERSION (mismos datos, lambda=20)",
          "p3_3b_malla_inversion.png")

---
## 3.4 — Resumen del Punto 3

Una tabla con los 36 resultados (CSV en `resultados_ert/punto3/`) y una figura que
resume, por geometría, cuánto contraste se recupera en cada variante.

### Lo que deja el Punto 3

1. **La forma importa menos que la dimensión.** El círculo y el triángulo (ambos 2.5D) dan casi lo mismo: ~45 % de recuperación. La esfera, con la **misma sección**, deja solo 10 % de anomalía en ρa y recupera ~12 %. Un cuerpo compacto 3D se ve **mucho más débil** que un cuerpo alargado del mismo corte. Interpretado en 2D, parece más chico y menos resistivo de lo que es.
2. **Electrodos:** es lo que más mejora la imagen (32 % → 53 %). Más electrodos en la misma línea dan más resolución, pero no más profundidad.
3. **λ:** controla cuánto contraste "se atreve" a mostrar la inversión. Se elige con χ² ≈ 1, no por la imagen más bonita.
4. **Mallas:** la de simulación debe tener un error menor que el ruido (aquí se cumple). La de inversión, una vez que es razonable, no cambia el resultado si se afina más.

In [ ]:
df3 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                    for r in resultados_p3])
df3.to_csv(os.path.join(OUT_P3, "metricas_punto3.csv"), index=False)

n_malos3 = int((~df3["valido"]).sum())
print("Casos NO validos: %d de %d" % (n_malos3, len(df3)))

# --- figura resumen: recuperacion por subpunto, una barra por geometria ---
subpuntos = list(dict.fromkeys(df3["subpunto"]))
colores3 = {"circular": "#2d6a9f", "triangular": "#b5523a", "esferico": "#4f8c5a"}
fig, axes = plt.subplots(1, len(subpuntos), figsize=(4.2 * len(subpuntos), 4),
                         sharey=True)
for ax, sp in zip(axes, subpuntos):
    d = df3[df3["subpunto"] == sp]
    variantes = list(dict.fromkeys(d["variante"]))
    xb = np.arange(len(variantes))
    for k, geo in enumerate(NOMBRES_GEO):
        y = [100 * d[(d["geometria"] == geo) & (d["variante"] == v)]
             ["recuperacion_log"].iloc[0] for v in variantes]
        ax.bar(xb + (k - 1) * 0.27, y, width=0.27, color=colores3[geo], label=geo)
    ax.set_xticks(xb)
    ax.set_xticklabels([v.replace("simulacion ", "sim. ")
                        .replace("inversion ", "inv. ") for v in variantes],
                       rotation=20, fontsize=9)
    ax.set_title(sp, fontsize=10)
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Contraste recuperado (% en log)")
axes[0].legend(fontsize=9)
fig.suptitle("Punto 3 - cuanto del contraste x10 recupera la inversion", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_P3, "p3_resumen.png"), dpi=150)
plt.show()

df3[["geometria", "subpunto", "variante", "n_datos", "n_celdas_inv",
     "error_malla_pct", "chi2", "rho_max_rec", "recuperacion_log"]]

---
## 7. Descargar los resultados

En Colab el sistema de archivos es **temporal**: cuando se cierra la sesión se
borra todo. Esta celda empaqueta la carpeta `resultados_ert/` (figuras + CSV, incluida la subcarpeta `punto3/`) en
un ZIP y lo descarga.

In [ ]:
import shutil

shutil.make_archive("resultados_ert", "zip", OUTDIR)
print("Comprimido resultados_ert.zip")
print("Contenido de %s/:" % OUTDIR)
for f in sorted(os.listdir(OUTDIR)):
    print("  -", f)

if EN_COLAB:
    from google.colab import files
    files.download("resultados_ert.zip")
else:
    print("\nEl ZIP quedo en:", os.path.abspath("resultados_ert.zip"))

---
## 8. Guía para redactar las conclusiones

No escribas las conclusiones como una lista de "en el caso A1 pasó X". Organízalas
alrededor de **qué controla la fidelidad de una imagen ERT**. Un orden que funciona:

**1. Sobre la profundidad de investigación (Exp. A).**
¿Hasta dónde ve este arreglo? Relaciona el resultado con el espaciamiento de
electrodos y la apertura de la línea. Menciona qué pasa con las estructuras que
caen por debajo de ese límite: no es que salgan mal, es que la inversión **no
tiene información** y rellena con el modelo inicial suavizado.

**2. Sobre el contraste (Exp. B).**
Documenta la asimetría conductivo/resistivo con los números de
`recuperacion_log`. Explica el mecanismo físico (la corriente busca el camino de
menor resistencia). Discute el límite de detectabilidad: ¿con contraste ×2 y ruido
del 1 %, la anomalía es distinguible?

**3. Sobre la geometría (Exp. C).**
Distingue entre la pérdida de resolución con la profundidad (la base del dique no
se ve), la sensibilidad lateral del dipolo‑dipolo (el dique se ubica bien en x) y
el principio de equivalencia (la capa delgada resistiva se diluye). Señala
explícitamente los artefactos y di que son del método, no del subsuelo.

**4. Sobre λ (Exp. D).**
Este es el que amarra todo: demuestra que **la imagen depende de una decisión del
intérprete**, no solo de los datos. Usa la curva L y el criterio χ²≈1.

**5. El hilo conductor para cerrar.**
La inversión no reconstruye el subsuelo: reconstruye *el subsuelo más suave
compatible con los datos*. Todo lo que observaste —contrastes subestimados, bordes
difusos, pérdida de resolución con la profundidad, artefactos— es consecuencia
directa de eso, no un error de la corrida. De ahí se sigue la recomendación
práctica: una sección ERT nunca debe interpretarse sin saber qué arreglo, qué λ y
qué nivel de ruido la produjeron.

---
## 9. Preguntas que vale la pena anticipar (con respuesta corta)

Intenta responderlas tú antes de leer la respuesta.

<details><summary><b>¿Por qué ρa negativa es imposible y por qué aparece de todos modos?</b></summary>

$\rho_a = K\,\Delta V / I$. Físicamente es positiva, pero en las cuádruplas más
abiertas $K$ es enorme y $\Delta V$ es diminuto. Basta que el ruido absoluto
(aquí 1 µV) sea comparable a $\Delta V$ para que el voltaje medido cambie de
signo, y $K$ lo amplifica. Por eso esos datos se eliminan antes de invertir.
En esta configuración no aparece ninguno, pero el filtro debe estar siempre.
</details>

<details><summary><b>¿Por qué la malla de inversión no puede ser la misma del modelo verdadero?</b></summary>

Es el **crimen inverso** (*inverse crime*). Si se invierte con la misma malla (y el
mismo solver) que generó los datos, la inversión tiene ventajas imposibles en la
realidad: las fronteras de las celdas coinciden con las del objeto y el error de
discretización se cancela. Los resultados salen artificialmente buenos. En campo
el subsuelo no viene alineado con tu malla.
</details>

<details><summary><b>Si χ² < 1, ¿el resultado es mejor o peor?</b></summary>

Peor, aunque parezca mejor. χ² < 1 significa que el modelo ajusta los datos
**más allá de su error**: está reproduciendo ruido. La estructura extra que
aparece no está respaldada por los datos. En el Exp. D, λ = 1 da χ² = 0.55 y más
contraste: ese contraste extra no se puede defender.
</details>

<details><summary><b>¿Cómo cambiarían las conclusiones con Wenner o Schlumberger?</b></summary>

Wenner tiene mejor relación señal/ruido y mejor resolución **vertical** (capas
horizontales), pero menos cobertura lateral y menos profundidad de investigación
relativa. Schlumberger está en un punto intermedio. Dipolo‑dipolo es el más
sensible a cambios **laterales** (diques, cavidades) pero tiene señales más
débiles, así que es más sensible al ruido. Se puede probar cambiando
`ESQUEMA = 'wa'` (Wenner) o `'slm'` (Schlumberger) en la configuración.
</details>

<details><summary><b>¿Por qué se usa la media geométrica y la escala log?</b></summary>

La resistividad varía en órdenes de magnitud (de 1 a 10 000 Ω·m), y lo que
importa físicamente son los **cocientes**: ×10 es el mismo "salto" sea de 5 a 50
o de 50 a 500. La inversión misma trabaja con $\ln\rho$. Una media aritmética la
dominarían los valores altos.
</details>

<details><summary><b>¿Qué es la profundidad de investigación y cómo la estimo?</b></summary>

Es la profundidad hasta la cual los datos todavía controlan el modelo. Una regla
práctica para dipolo‑dipolo es ~0.2 × la apertura total (aquí ~6 m). Más
rigurosamente se estima con la **cobertura/sensibilidad** acumulada
(`mgr.coverage()` en PyGIMLi) o con el índice DOI de Oldenburg & Li: se invierte
dos veces con modelos iniciales distintos, y donde los resultados coinciden, los
datos mandan.
</details>